## Plan microsatellite analysis

*Goal*
- Create plates from the extracted samples, that will be used for the microsatellite analysis
- Probably, the same plates would be used for sequencing, so probably want to do something that makes sense for both

In [241]:
import os
import string
import itertools
import pandas as pd
from dataclasses import dataclass, fields

### Settings

In [242]:
save_outputs = True
output_dir = "../metadata/2026-01-19_design-microsat-plates"
os.makedirs(output_dir, exist_ok=True)

### Load data

In [243]:
ID = "sample_id"
df_zmb = pd.read_csv("../metadata/2025-11-28_select-mpiib-samples-jh/table.zambia_selectionset.complete.csv", dtype={ID: str})
df_extract = pd.read_csv("../metadata/2025-12-11_mpiib-extractions-km/zambia_isolation_list.csv", dtype={ID: str})

### Check for differences

In [244]:
df_zmb.shape, df_extract.shape

((300, 11), (284, 4))

- 16 fewer extracted than selected, at least

In [245]:
print(f"Selected for MPIIB, but not in extraction set:\n{','.join(set(df_zmb[ID]).difference(df_extract[ID]))}.\n")
print(f"Extracted at MPIIB, but not selected:\n{', '.join(set(df_extract[ID]).difference(df_zmb[ID]))}")

Selected for MPIIB, but not in extraction set:
1036155,1045440,1012546,8021942,8021020,8021943,8023250,8010995,1036152,8023990,1036154,1045433,5053645,1036159,1036156,8021937,1053943,1036158,1053971,1036157,1036151,1036153.

Extracted at MPIIB, but not selected:
1045445, 1045435, 5053643, 8011963, 8021018, 8021025


### Focus on organising extracted

In [246]:
df_extract

,sample_id,extraction_id,extraction_date,notes
0,8010683,Z001,04.12.25,NaN
1,8034059,Z002,04.12.25,NaN
2,8012421,Z003,04.12.25,NaN
3,8032055,Z004,04.12.25,NaN
4,5053837,Z005,04.12.25,NaN
...,...,...,...,...
279,8021027,Z280,12.12.25,NaN
280,1012412,Z281,12.12.25,NaN
281,2030496,Z282,12.12.25,NaN
282,1012544,Z283,12.12.25,NaN


In [247]:
df_joint = pd.merge(
    left=df_extract[[ID, "extraction_id"]],
    right=df_zmb,
    on=ID,
    how='inner'
)

In [248]:
df_joint

,sample_id,extraction_id,set_type,district,province,expt_name,barcode,gt_A724E,dp_A724E,wsaf_A724E,clonality,carrier_A724E
0,8010683,Z001,representative,chipata,eastern,2025-10-08_SLMM088_HRP2_2024_Batch20,barcode52,0.0,NaN,0.000000,clonal,False
1,8034059,Z002,clonal,sesheke,western,2025-10-30_SLMM092_HRP2_2024_Batch24,barcode66,2.0,2763.0,1.000000,clonal,True
2,8012421,Z003,representative,vubwi,eastern,2025-10-08_SLMM089_HRP2_2024_Batch21,barcode28,0.0,NaN,0.000000,clonal,False
3,8032055,Z004,representative,manyinga,northwestern,2024-10-30_SLMM060_MIS2024Batch2,barcode34,2.0,398.0,1.000000,clonal,True
4,5053837,Z005,representative,solwezi,northwestern,2025-07-02_SLMM077_HRP2-32024_Batch10,barcode66,0.0,582.0,0.000000,clonal,False
...,...,...,...,...,...,...,...,...,...,...,...,...
273,8021027,Z280,representative,kasempa,northwestern,2025-04-03_SLMM071_HRP22024_Batch4,barcode52,0.0,4726.0,0.000132,clonal,False
274,1012412,Z281,clonal,zambezi,northwestern,2025-07-02_SLMM077_HRP2-32024_Batch10,barcode70,2.0,582.0,1.000000,clonal,True
275,2030496,Z282,representative,solwezi,northwestern,2025-04-04_SLMM072_HRP22024_Batch5,barcode11,2.0,577.0,0.998313,polyclonal,True
276,1012544,Z283,representative,zambezi,northwestern,2025-07-02_SLMM077_HRP2-32024_Batch10,barcode69,2.0,354.0,1.000000,clonal,True


### Small explore / sanity check

In [249]:
pd.crosstab(
    df_joint["province"],
    df_joint["gt_A724E"],
    margins=True
)

gt_A724E,0.0,1.0,2.0,All
province,,,,
eastern,51,0,0,51
muchinga,7,0,1,8
northwestern,48,20,33,101
western,61,24,33,118
All,167,44,67,278


In [250]:
df_joint.query("clonality == 'clonal' and set_type == 'representative'").shape 
# so we have 118 randomly sample clonals
# but.. wether you sample randomly and pick clonals, or pick clonals and randomly sample... is it any different?

(118, 12)

In [251]:
_df = df_joint.query("clonality == 'clonal' and set_type == 'representative'")

In [252]:
pd.crosstab(_df["province"],
            _df["gt_A724E"], margins=True)

gt_A724E,0.0,1.0,2.0,All
province,,,,
eastern,30,0,0,30
muchinga,2,0,1,3
northwestern,17,3,19,39
western,25,2,19,46
All,74,5,39,118


### Design plates for these

- Too complicated to also plan for WGS now, let's just focus on microsatellites
- I could separate the probable clonal and polyclonal samples, as clonal will be much more useful for analysis later

*Goal*
- Split into 2-3 plates
- Add controls
- Give well information


*Idea*
- For both microsat and WGS, clonal samples are easier to analyse
- Thus it makes sense to split into clonal / polyclonal plates, as focussing on clonal first in lab would save money / time, if ultimately we fail to manage with the polyclonal sample analysis
- Otherwise, we can just keep things random
  

**Define wells and controls**

In [285]:
# WELLS = [f"{r}{c}" for r,c in itertools.product(string.ascii_uppercase[:8], range(1, 13))]
# len(WELLS)

In [286]:
# Better order for Karolina
WELLS = [
    f"{r}{c}"
    for c in range(1, 13)
    for r in string.ascii_uppercase[:8]
]   
len(WELLS)

96

In [287]:
@dataclass
class Control:
    sample_id: str
    extraction_id: str = None
    set_type: str = None
    district: str = None
    province: str = None
    carrier_A724E: bool = False

In [288]:
df_controls = pd.DataFrame([Control("3D7_10K"), Control("Dd2_10K"), Control("NTC")])

In [289]:
df_controls

,sample_id,extraction_id,set_type,district,province,carrier_A724E
0,3D7_10K,None,None,None,None,False
1,Dd2_10K,None,None,None,None,False
2,NTC,None,None,None,None,False


**Split into clonal / polyclonal**
- We will avoid using polyclonal samples for microsatellite analysis, as you cannot analyse them

In [290]:
df_joint_clonal = df_joint.query("clonality == 'clonal' and gt_A724E != 1")
_s = df_joint_clonal.sample_id.tolist()
df_joint_poly = df_joint.query("sample_id not in @_s")

In [291]:
df_joint.shape, df_joint_clonal.shape, df_joint_poly.shape

((278, 12), (166, 12), (112, 12))

In [292]:
pd.crosstab(df_joint_clonal["carrier_A724E"],
            df_joint_clonal["province"], margins=True)

province,eastern,muchinga,northwestern,western,All
carrier_A724E,,,,,
False,30,2,33,46,111
True,0,1,26,28,55
All,30,3,59,74,166


In [293]:
pd.crosstab(df_joint_clonal["carrier_A724E"],
            df_joint_clonal["set_type"], margins=True)

set_type,clonal,representative,All
carrier_A724E,,,
False,37,74,111
True,16,39,55
All,53,113,166


#### Create clonal plates
- If we have 6 controls, that is 90 samples per plate
- With 166 clonal samples, that is two plates

In [294]:
n_samples = 90

In [295]:
df_joint_clonal.shape

(166, 12)

In [296]:
df_plate1 = pd.concat([df_controls, df_joint_clonal.iloc[:n_samples], df_controls])[[f.name for f in fields(Control)]]
df_plate1.shape

(96, 6)

In [297]:
df_plate1.insert(0, "plate", "C01")
df_plate1.insert(1, "well", WELLS[:df_plate1.shape[0]])

In [298]:
df_plate2 = pd.concat([df_controls, df_joint_clonal.iloc[n_samples:], df_controls])[[f.name for f in fields(Control)]]
df_plate2.shape

(82, 6)

In [299]:
df_plate2.insert(0, "plate", "C02")
df_plate2.insert(1, "well", WELLS[:df_plate2.shape[0]])

*Sanity check*

In [300]:
df_joint_clonal.shape[0] == (df_plate1.shape[0] + df_plate2.shape[0]) - (df_controls.shape[0])*4 

True

In [301]:
set(df_plate1.sample_id).intersection(set(df_plate2.sample_id)) # good

{'3D7_10K', 'Dd2_10K', 'NTC'}

*Characterise*

In [302]:
total_a724e = df_plate1.carrier_A724E.sum() + df_plate2.carrier_A724E.sum()
total = df_joint_clonal.shape[0] 

In [303]:
total_a724e

np.int64(55)

In [304]:
total # so we have only 55 carring the mutation, not so many

166

In [306]:
df_plate1.head()

,plate,well,sample_id,extraction_id,set_type,district,province,carrier_A724E
0,C01,A1,3D7_10K,None,None,None,None,False
1,C01,B1,Dd2_10K,None,None,None,None,False
2,C01,C1,NTC,None,None,None,None,False
0,C01,D1,8010683,Z001,representative,chipata,eastern,False
1,C01,E1,8034059,Z002,clonal,sesheke,western,True


**Write**

In [305]:
if save_outputs:
    df_plate1.to_csv(f"{output_dir}/table.microsat_plate1.C01.csv", index=False)
    df_plate2.to_csv(f"{output_dir}/table.microsat_plate2.C02.csv", index=False)

#### Create polyclonal plates
- We actually don't need these yet

In [140]:
df_joint_poly.shape

(112, 12)

## Deciding on how to pool

Option 1:
- Do all samples for 1 primer
- Do all samples for 2 primer
- Pool those two, read

Option 2:
- 


In [200]:
df_joint_poly.query("province == 'eastern'")

,sample_id,extraction_id,set_type,district,province,expt_name,barcode,gt_A724E,dp_A724E,wsaf_A724E,clonality,carrier_A724E
5,8013476,Z006,representative,chipata,eastern,2025-10-14_SLMM090_HRP2_2024_Batch22,barcode40,0.0,NaN,0.0,polyclonal,False
8,8023679,Z009,representative,katete,eastern,2025-10-14_SLMM091_HRP2_2024_Batch23,barcode43,0.0,NaN,0.0,polyclonal,False
18,8033412,Z019,representative,mambwe,eastern,2025-10-30_SLMM093_HRP2_2024_Batch25,barcode65,0.0,NaN,0.0,polyclonal,False
22,8022819,Z023,representative,chipangali,eastern,2025-10-14_SLMM090_HRP2_2024_Batch22,barcode83,0.0,NaN,0.0,polyclonal,False
54,8033426,Z056,representative,mambwe,eastern,2025-10-30_SLMM093_HRP2_2024_Batch25,barcode75,0.0,NaN,0.0,polyclonal,False
72,8013555,Z074,representative,chipangali,eastern,2025-05-08_SLMM074_MIS2024_Batch6,barcode08,0.0,NaN,0.0,polyclonal,False
76,8022315,Z078,representative,lundazi,eastern,2025-05-08_SLMM074_MIS2024_Batch6,barcode43,0.0,NaN,0.0,polyclonal,False
78,8013686,Z080,representative,chipata,eastern,2025-10-14_SLMM090_HRP2_2024_Batch22,barcode67,0.0,NaN,0.0,polyclonal,False
81,8033355,Z083,representative,mambwe,eastern,2025-10-30_SLMM093_HRP2_2024_Batch25,barcode32,0.0,NaN,0.0,polyclonal,False
145,8022865,Z147,representative,chipangali,eastern,2025-10-14_SLMM091_HRP2_2024_Batch23,barcode17,0.0,NaN,0.0,polyclonal,False


In [201]:
for_karolina = ["Z006", "Z0074", "Z210"]